# Clase 6 · Fauna de la Orinoquía (gemelo, trabajo autónomo)

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelve cada uno en su trabajo autónomo, porque el día de la clase hay parcial. Es el gemelo de *La tierra desde arriba*: la misma pregunta sobre el cambio de dominio, con fotos de cámaras trampa.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Medir por cuenta propia cuánto pierde un clasificador de especies cuando se usa en cámaras que nunca vio, detectar ese cambio de dominio y convertirlo en una regla sobre cuáles fotos etiqueta el modelo y cuáles revisa una persona. |
| **Enfoque** | Machine Learning. |
| **Resultados de aprendizaje** | 1 (formular), 2 (seleccionar y preparar datos), 3 (flujo reproducible), 4 (implementar y evaluar), 5 (comunicar) y 6 (reflexión ética, técnica y operativa). |
| **Uso de IA** | El código lo escribe Claude y ustedes lo auditan. Deben poder explicar sin ayuda por qué se parte por cámara, cuánto cambió el resultado frente a partir al azar, qué significa la cobertura de una regla de confianza y qué promesa le pueden hacer al cliente. |
| **Entregable** | El cuaderno completo y corrido en su portafolio de GitHub, con las predicciones escritas, las respuestas, la auditoría, la decisión en tres frases y la tabla de la variante llenas. |

**Cómo se trabaja.** Igual que en *La roya*: cada celda de código trae solo un comentario que dice qué debe lograr. Péguenle a Claude el bloque «Lo que Claude necesita saber», pídanle el código por secciones, léanlo, córranlo y respondan las preguntas con sus palabras.

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

In [ ]:
CONFIANZA = 0.80        # el modelo etiqueta solo las fotos en las que su confianza llega a este valor. La cambiarán al final.
SEMILLA = 0

import time
from taller.clase05_imagenes import imagenes, rasgos_guardados, guardar_rasgos

### Lo que Claude necesita saber

> Trabajo en un cuaderno de Colab sin GPU. Ya están importados `numpy as np`, `pandas as pd`, `matplotlib.pyplot as plt` y `time`, y definidas las constantes `CONFIANZA` y `SEMILLA`.
>
> `tabla = descargar("orinoquia")` devuelve un DataFrame con una fila por foto de cámara trampa y estas columnas: `archivo`, `especie` (ocho valores, en inglés, por ejemplo `lowland_tapir`), `camara` (48 cámaras, por ejemplo `N25`), `seq_id` (identificador de la ráfaga; hay una sola foto por ráfaga), `frame_num`, `seq_num_frames`, `fecha` y `ruta_lila`.
>
> `carpeta = imagenes("orinoquia", tabla["archivo"])` devuelve la carpeta (`pathlib.Path`) con las fotos en JPEG, reducidas a 256 píxeles de lado corto. La foto de una fila es `carpeta / fila["archivo"]`.
>
> `rasgos_guardados(nombre, filas)` devuelve una matriz de numpy ya calculada o `None`, y `guardar_rasgos(nombre, matriz)` la guarda.
>
> Quiero usar ResNet18 de torchvision, preentrenada en ImageNet, como extractor fijo de características, y encima modelos de scikit-learn. Todo debe correr en CPU en pocos minutos. Escribe código corto, con comentarios en español, y no uses datos que no sean los de la tabla.

## 1. Formulación

Imagínense un grupo de biólogos que estudia la fauna de dos reservas naturales privadas del Meta, El Rey Zamuro y Las Unamas. Pusieron cincuenta cámaras trampa, separadas un kilómetro entre sí, de enero a julio de 2020. Una cámara trampa dispara una ráfaga de fotos cuando algo se mueve. El resultado son más de cien mil fotos que alguien tiene que mirar.

El grupo quiere un modelo que diga qué especie hay en cada foto. Y quiere usarlo la próxima temporada, cuando las cámaras estén en **otros sitios**.

1. **Predecir.** Dada una foto, ¿qué especie aparece?
2. **Explicar.** ¿El modelo reconoce al animal o reconoce el sitio donde está la cámara?
3. **Decidir.** En cámaras nuevas, ¿cuáles fotos etiqueta el modelo solo y cuáles pasan a un biólogo?

**El dato.** *Orinoquía Camera Traps* está publicado en LILA BC (Vélez y colaboradores, *Methods in Ecology and Evolution*, 2023), con la licencia Community Data License Agreement en su variante permisiva. Para el curso preparé un subconjunto: ocho especies, **una sola foto por ráfaga** (la del medio) y hasta 250 ráfagas por especie. Son 1.876 fotos de 48 cámaras. Quienes publicaron el conjunto retiraron las fotos en las que aparecen personas.

**El dominio.** Aquí cada cámara es un dominio: tiene su fondo, su altura, su luz. El cambio de dominio es pasar de las cámaras de entrenamiento a cámaras nuevas, $P_o(x,y)\neq P_d(x,y)$.

**La métrica.** Exactitud y exactitud balanceada sobre las $K=8$ especies,

$$\text{exactitud}=\frac{1}{n}\sum_{i=1}^{n}\mathbb{1}\{\hat y_i=y_i\},\qquad \text{exactitud balanceada}=\frac{1}{K}\sum_{k=1}^{K}\frac{\#\{i:\ \hat y_i=y_i=k\}}{\#\{i:\ y_i=k\}}.$$

**La regla de decisión.** El modelo le da a cada foto una confianza $c(x)=\max_k\hat p_k(x)$. Con un umbral $\tau$ (`CONFIANZA`), el modelo etiqueta solo las fotos con $c(x)\ge\tau$ y manda el resto al biólogo. Dos números describen la regla:

$$\text{cobertura}(\tau)=\frac{1}{n}\sum_{i}\mathbb{1}\{c(x_i)\ge\tau\},\qquad \text{exactitud selectiva}(\tau)=\frac{\sum_i \mathbb{1}\{c(x_i)\ge\tau\}\,\mathbb{1}\{\hat y_i=y_i\}}{\sum_i \mathbb{1}\{c(x_i)\ge\tau\}}.$$

La cobertura es el trabajo que el modelo le quita al biólogo. La exactitud selectiva es la calidad de lo que el modelo etiqueta solo.

### Predicciones escritas

- ¿Qué exactitud tendrá el modelo si se parte al azar por foto? **Mi apuesta:**
- ¿Y en cámaras que nunca vio? **Mi apuesta:**
- Con ocho especies, ¿con qué acierto se podrá adivinar la **cámara** (entre 48) a partir de las características? **Mi apuesta:**

## 2. Datos

### 2.1 Descargar y contar

La descarga trae 1.876 fotos de Google Cloud y tarda cerca de un minuto.

In [ ]:
# TAREA 1. Descarguen la tabla con descargar("orinoquia") y midan el tiempo. Cuenten fotos por especie,
# número de cámaras y fotos por cámara (mínimo, mediana, máximo). Comprueben que cada seq_id aparece una sola vez.

In [ ]:
# TAREA 2. Tabla cruzada de cámara contra especie, dibujada como mapa de calor (cámaras en filas, ordenadas
# por nombre). Calculen además, para cada cámara, qué fracción de sus fotos es de su especie más frecuente.

**Preguntas.**

- ¿Las especies se reparten parejo entre cámaras? ¿Qué componente de $P(x,y)$ cambia entre una cámara y otra según esta tabla? **Mi respuesta:**
- Los nombres de las cámaras empiezan por `A`, `M` o `N`. La página del conjunto no dice qué significan esas letras. ¿Qué podrían ser, y por qué no conviene suponerlo? **Mi respuesta:**
- ¿Por qué el subconjunto tiene una sola foto por ráfaga? Relaciónenlo con las lesiones de la clase 5. **Mi respuesta:**

In [ ]:
# TAREA 3. Obtengan la carpeta con imagenes("orinoquia", tabla["archivo"]). Muestren una foto de cada especie
# (fila de 8) y, debajo, 8 fotos de la cámara con más fotos, con la especie en el título.

In [ ]:
# TAREA 4. Una foto nocturna de cámara trampa es infrarroja y sale en grises. Calculen para cada foto si es
# gris (los tres canales casi iguales) y reporten la fracción de fotos grises por especie.

**Pregunta.** ¿Qué especies salen casi siempre de noche y cuáles de día? Si el modelo aprendiera solo «gris o color», ¿cuánto adelantaría? **Mi respuesta:**

### 2.2 Características y exploración

In [ ]:
# TAREA 5. Extraigan las 512 características de cada foto con ResNet18 preentrenada (sin la última capa, en
# modo de evaluación, con la transformación que acompaña a los pesos), en el orden de la tabla. Usen
# rasgos_guardados("orinoquia_resnet18", len(rutas)) y guardar_rasgos. Midan el tiempo.

In [ ]:
# TAREA 6. PCA de las características estandarizadas: dibujen los dos primeros componentes coloreados por
# especie y, al lado, coloreados por cámara (solo las 6 cámaras con más fotos). Después, para cada foto,
# busquen su vecina más cercana y reporten en qué porcentaje es de la misma cámara y de la misma especie,
# junto con lo que saldría al azar.

**Pregunta.** ¿Qué reconoce mejor la red, la cámara o la especie? ¿Qué anticipa eso para un modelo que se examine en cámaras nuevas? **Mi respuesta:**

## 3. Modelado

### 3.1 La partición por cámara

Tres tramos sorteando **cámaras**: 60 % para entrenar, 20 % para validar y 20 % para la prueba. Así la validación y la prueba son siempre cámaras nuevas para el modelo, que es la situación del cliente.

In [ ]:
# TAREA 7. Partan los índices en E, V y P sorteando cámaras (60/20/20) con SEMILLA. Impriman fotos y cámaras
# por tramo y comprueben con un assert que ninguna cámara aparece en dos tramos.

### 3.2 Línea base y cinco modelos

La línea base tiene dos versiones: responder siempre la especie más frecuente del entrenamiento, y una regresión logística sobre seis números de color por foto (promedio y desviación de cada canal). Encima de las características compiten los cinco clasificadores de la demostración.

In [ ]:
# TAREA 8. Calculen las dos líneas base y entrenen los cinco modelos (logística, SVM, k vecinos, bosque,
# boosting sobre componentes principales) en E. Reporten exactitud y exactitud balanceada en V y en P en
# una tabla ordenada por la exactitud balanceada de validación, con una gráfica de barras.

### 3.3 La trampa: el examen en casa

Ahora midan lo que habría salido con una partición al azar por foto, que mezcla las cámaras. Ese es el examen en casa: el modelo ya conoce el fondo de cada cámara.

In [ ]:
# TAREA 9. Repitan 30 veces (semillas 0 a 29), para la regresión logística y para un vecino (k = 1): partir por
# cámara y partir al azar por foto, con 20 % en prueba. Guarden la exactitud en cada caso, muestren la media y
# la desviación estándar y un diagrama de caja.

**Preguntas.** Con los números de su tabla.

- ¿Cuántos puntos de exactitud separan el examen en casa del examen en cámaras nuevas? ¿La diferencia es mayor que el ruido entre sorteos? **Mi respuesta:**
- ¿Cuál de los dos números le pueden prometer al grupo de biólogos para la próxima temporada? **Mi respuesta:**
- La desviación estándar es mayor al partir por cámara. ¿Por qué? **Mi respuesta:**

### 3.4 Detectar y explicar

Como en la demostración, hay señales que no necesitan etiquetas de especie. Aquí el «clasificador de dominio» adivina la cámara.

In [ ]:
# TAREA 10. Entrenen una regresión logística que adivine la CÁMARA a partir de las características, con una
# partición al azar por foto (75/25). Reporten su exactitud y compárenla con responder siempre la cámara
# más frecuente.

In [ ]:
# TAREA 11. Con el modelo ganador, en validación (cámaras nuevas): acierto por especie y matriz de confusión
# dibujada como mapa de calor, normalizada por filas.

In [ ]:
# TAREA 12. El remedio: ¿ayuda tener más cámaras en el entrenamiento? Con la regresión logística, entrenen
# con 4, 8, 16 y todas las cámaras de E (sorteadas, 10 repeticiones cada una) y midan la exactitud en V.
# Dibujen la exactitud contra el número de cámaras y anoten cuántas fotos de entrenamiento hubo en promedio.

**Preguntas.**

- ¿La red sabe en qué cámara está cada foto? ¿Qué significa eso para la pregunta de explicar? **Mi respuesta:**
- ¿Qué especies se confunden entre sí en cámaras nuevas? Miren fotos de esas especies antes de responder. **Mi respuesta:**
- Según la curva, ¿qué le recomendarían al grupo: más fotos por cámara o más cámaras? ¿La curva ya se aplanó? **Mi respuesta:**

## 4. Comunicación

### 4.1 Decidir: qué fotos etiqueta el modelo solo

Apliquen la regla de confianza con $\tau=$ `CONFIANZA`. Midan cobertura y exactitud selectiva en tres exámenes: en casa (una partición al azar), en validación y en prueba (cámaras nuevas). La incertidumbre de la prueba se mide con *bootstrap* **por cámara**, $\text{IC}_{95\%}=[\hat\theta^*_{(0{,}025)},\hat\theta^*_{(0{,}975)}]$, porque las fotos de una cámara no son independientes.

In [ ]:
# TAREA 13. Con un modelo que entregue probabilidades (el ganador si tiene predict_proba; si no, la logística):
# cobertura y exactitud selectiva con el umbral CONFIANZA en (a) una partición al azar 80/20, (b) validación
# y (c) prueba. Para la prueba, intervalos de 95 % por bootstrap de cámaras (1.000 réplicas). Dibujen además
# la curva de exactitud selectiva contra cobertura para todos los umbrales, en casa y en cámaras nuevas.

**Preguntas.**

- Con el mismo umbral, ¿qué cobertura y qué exactitud selectiva prometía el examen en casa y cuáles se cumplieron en cámaras nuevas? **Mi respuesta:**
- De cada 1.000 fotos de una cámara nueva, ¿cuántas etiqueta el modelo solo y cuántas de esas quedan mal etiquetadas? **Mi respuesta:**
- ¿El intervalo es ancho o angosto? ¿Cuántas cámaras tiene la prueba? **Mi respuesta:**

### La decisión en tres frases

Para quien dirige el grupo de biólogos.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Auditoría del código que les dio la IA

Comprueben estas tres cosas y escriban qué encontraron y en qué celda.

1. **Las cámaras no se mezclan.** Ninguna cámara aparece en dos tramos y hay un `assert` que lo comprueba. Revisen también que la partición por grupos no use `stratify`, que con grupos no hace lo que uno cree. **Lo que encontré:**
2. **El número que se promete es el de cámaras nuevas.** Busquen en cada tabla y en cada gráfica de la sección 4 con qué partición se calculó. El modelo y el umbral no se escogieron mirando la prueba. **Lo que encontré:**
3. **La confianza es lo que dice ser.** Comprueben en validación que, entre las fotos con confianza de 0,8 o más, la fracción de aciertos se parece a la confianza promedio. Si el modelo es demasiado seguro, la regla promete más de lo que cumple. **Lo que encontré:**

In [ ]:
# TAREA 14. Auditoría 3: en validación, partan las fotos en 5 grupos según la confianza del modelo y comparen
# la confianza promedio de cada grupo con su fracción de aciertos.

### 4.3 Bitácora de decisiones

Llenen las filas vacías. La primera va de ejemplo.

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Una foto por ráfaga | Todas las fotos de cada ráfaga | Las fotos de una ráfaga son casi iguales; con todas, el resultado se infla y el cálculo se multiplica |
| Ocho especies, sin fotos vacías | | |
| Hasta 250 ráfagas por especie | | |
| Partición por cámara | | |
| Exactitud balanceada para escoger el modelo | | |
| Regla de confianza con umbral fijo | | |
| *Bootstrap* por cámara | | |

### 4.4 Riesgos y mantenimiento

- El subconjunto no tiene fotos vacías ni especies raras, y en el conjunto completo una de cada cinco fotos está vacía. ¿Qué hará el modelo con una foto vacía o con un puma? ¿Qué hará la regla de confianza?
- Las reservas son privadas y las cámaras también fotografían personas. Quienes publicaron el dato retiraron esas fotos. ¿Qué deberían hacer ustedes con las fotos de personas si montan este sistema para la próxima temporada?
- La ubicación de especies amenazadas es información sensible. ¿Qué parte de una tabla como esta no publicarían?
- Las cámaras nuevas de la prueba están en las mismas dos reservas y en los mismos meses. ¿Qué cambio de dominio queda sin medir?

### 4.5 Cambien una línea

Vuelvan a la celda del comienzo, cambien `CONFIANZA = 0.80` por `CONFIANZA = 0.95` y corran todo de nuevo. Llenen la tabla.

| En la prueba (cámaras nuevas) | $\tau=0{,}80$ | $\tau=0{,}95$ |
|---|---|---|
| Cobertura | | |
| Exactitud selectiva | | |
| Intervalo de la exactitud selectiva | | |
| Fotos que revisa el biólogo, de cada 1.000 | | |
| Fotos mal etiquetadas por el modelo, de cada 1.000 | | |

Con la tabla llena, respondan en tres líneas: ¿qué umbral le recomiendan al grupo y qué gana y qué pierde con él? **Mi respuesta:**

Dejen el cuaderno entregado con `CONFIANZA = 0.95` corrido y la tabla completa.

### 4.6 Procedencia y entrega

In [ ]:
# TAREA 15. Muestren la procedencia del dato con procedencia()["orinoquia"].

**Para entregar:** en Colab, menú *Archivo*, *Guardar una copia en GitHub*, y escojan su repositorio de portafolio, carpeta `clase06/`. Un cuaderno que no corre de principio a fin desde cero no se califica: antes de guardar usen *Entorno de ejecución*, *Reiniciar sesión y ejecutar todo*.